In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from IPython.display import Image
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator

load_dotenv()

model = ChatOpenAI(model='gpt-4o-mini')

#create a schema

class EvaluationSchema(BaseModel):

    feedback: str = Field(description='detailed feedback for the eassay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

structure_model = model.with_structured_output(EvaluationSchema)

essay = """provide your sample essay"""

prompt = 'Evaluate the language quality of the essay and provide a feedback and assign a score'

structure_model.invoke(prompt)

class ExamState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    #this is the reducer , here it means scores are appended in the list e.g{6,9,7}
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float


def language(state:ExamState):

    prompt = f'Evaluate the language quality of the essay and provide a feedback and assign a score out of 10 \n{state['essay']}'
    output = structure_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores':[output.score]}

def analysis(state:ExamState):

    prompt = f'Evaluate the analysis score of the essay and provide a feedback and assign a score out of 10 \n{state['essay']}'
    output = structure_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores':[output.score]}

def thought(state:ExamState):

    prompt = f'Evaluate the clarity of thought of the essay and provide a feedback and assign a score out of 10 \n{state['essay']}'
    output = structure_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores':[output.score]}

def final_evaluation(state:ExamState):

    #summary feedback
    prompt = f'Based on the following feedback generate a summary feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]}\n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content
    #calculate average
    avg_score = sum(state['individual_scores']/len(state['individual_scores']))

    return {'overall_feedback':overall_feedback, 'avg_score':avg_score}

graph = StateGraph(ExamState)

graph.add_node('evaluate_language', language)
graph.add_node('evaluate_analysis', analysis)
graph.add_node('evaluate_thought', thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')


graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()


intial_state ={
    'essay':essay
}

final_state = workflow.invoke(intial_state)



